# 00 · Overview & setup — CFTR Variant Toolkit

A beginner-friendly, **honest** tour of the computational tools used to interpret
CFTR variants. Each notebook takes one tool (or tool family), explains *what it
is*, *what its score means*, *the decision threshold and why*, and *how to get the
real data* — then notebook 12 combines them into the A1/A2 worklists and notebook
08 covers the methodology pitfalls.

This notebook just checks your setup and shows the honest data-provenance map.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
print("Python :", sys.version.split()[0])
print("toolkit:", pathlib.Path(tk.__file__).name)
print("cache  :", tk.CACHE_DIR, "->", "FOUND" if tk.CACHE_DIR.exists() else "MISSING")

Python : 3.13.14
toolkit: toolkit.py
cache  : E:\cftr_variant_toolkit\_tmp_fetch -> FOUND


## The one thing to understand before anything else: REAL vs DEMO

This toolkit is deliberately transparent about what is a *real* download/prediction
and what is a small *hand-curated demo table* shipped so the notebooks run
anywhere. **Never quote a DEMO number as a finding.**

In [2]:
prov = pd.DataFrame([
  ("gnomAD v4 (allele freq)", "REAL",  "~2,466 missense / ~1,085 non-coding", "01"),
  ("AlphaMissense",           "REAL",  "genome-wide (all CFTR missense)",     "02"),
  ("EVE",                     "REAL",  "~26,809 CFTR variants",               "03"),
  ("ESM1b",                   "REAL",  "~28,120 CFTR variants (saturation)",  "04"),
  ("REVEL",                   "REAL",  "~10,127 CFTR (coord-keyed)",          "05"),
  ("PrimateAI",               "REAL",  "~1,976 CFTR (dbNSFP subset)",         "06"),
  ("ClinVar",                 "REAL",  "genome-wide",                         "07"),
  ("CFTR2 (30 Jan 2026)",     "REAL",  "~2,097 variants / 780 missense keys", "08"),
  ("SpliceAI",                "REAL",  "~566k CFTR SNVs (Illumina v1.3)",     "09"),
  ("Pangolin",                "DEMO",  "9 curated splice variants",           "10"),
  ("CADD",                    "REAL",  "live per-variant API",                "11"),
], columns=["source","status","coverage","notebook"])
prov

,source,status,coverage,notebook
0,gnomAD v4 (allele freq),REAL,"~2,466 missense / ~1,085 non-coding",01
1,AlphaMissense,REAL,genome-wide (all CFTR missense),02
2,EVE,REAL,"~26,809 CFTR variants",03
3,ESM1b,REAL,"~28,120 CFTR variants (saturation)",04
4,REVEL,REAL,"~10,127 CFTR (coord-keyed)",05
5,PrimateAI,REAL,"~1,976 CFTR (dbNSFP subset)",06
6,ClinVar,REAL,genome-wide,07
7,CFTR2 (30 Jan 2026),REAL,"~2,097 variants / 780 missense keys",08
8,SpliceAI,REAL,~566k CFTR SNVs (Illumina v1.3),09
9,Pangolin,DEMO,9 curated splice variants,10


In [3]:
# quick proof the REAL loaders work against the cache
print("gnomAD missense :", len(tk.load_gnomad_missense()), "(REAL)")
print("AlphaMissense   :", len(tk.load_alphamissense()), "(REAL, genome-wide)")
print("ClinVar         :", len(tk.load_clinvar()), "(REAL)")
print("CFTR2           :", len(tk.load_cftr2()), "(REAL, 30 Jan 2026 release)")
print("EVE             :", len(tk.load_eve()), "(REAL, ~26.8k variants)")
print("ESM1b           :", len(tk.load_esm1b()), "(REAL, ~28k saturation)")
print("REVEL           :", len(tk.load_revel()), "(REAL, ~10k coord-keyed)")
print("PrimateAI       :", len(tk.load_primateai()), "(REAL, ~2k dbNSFP subset)")
print("SpliceAI        :", len(tk.load_spliceai()), "(REAL, ~566k CFTR SNVs)")

gnomAD missense : 2466 (REAL)
AlphaMissense   : 8597 (REAL, genome-wide)
ClinVar         : 2801 (REAL)
CFTR2           : 2097 (REAL, 30 Jan 2026 release)
EVE             : 26809 (REAL, ~26.8k variants)
ESM1b           : 28120 (REAL, ~28k saturation)
REVEL           : 10127 (REAL, ~10k coord-keyed)
PrimateAI       : 1976 (REAL, ~2k dbNSFP subset)


SpliceAI        : 566106 (REAL, ~566k CFTR SNVs)


## How to read this toolkit

One tool per notebook (03-11); the integration notebook (12) is where they are combined and compared.

| Notebook | Tool | Real? |
|---|---|---|
| **01** | gnomAD — population allele frequency | REAL |
| **02** | AlphaMissense — the one real genome-wide predictor | REAL |
| **03** | EVE — unsupervised evolutionary model | **REAL** |
| **04** | ESM1b — protein language model (backwards scale) | **REAL** |
| **05** | REVEL — supervised ensemble (+ circularity) | **REAL** |
| **06** | PrimateAI — semi-supervised | **REAL** (subset) |
| **07** | ClinVar — crowd-sourced clinical truth set | REAL |
| **08** | CFTR2 — functional truth set | **REAL** |
| **09** | SpliceAI — splice deltas (all CFTR SNVs) | **REAL** |
| **10** | Pangolin — independent splice model | demo |
| **11** | CADD — real, live deleteriousness score | REAL (live) |
| **12** | **Integration** — reproduce A1/A2 + cross-tool comparisons | mixed |
| **13** | **Circular reasoning & training leakage** — the methodology | — |

Recommended order: 01 → 13 in sequence. If you only read two: **12** (what the
numbers really are, plus the tool-vs-tool comparisons) and **13** (why to be careful).

## Variant *consequence* — the vocabulary used throughout

A variant's **consequence** is what it does to the gene product, assigned by
mapping it onto the transcript (Ensembl VEP / Sequence Ontology terms; McLaren
et al. 2016, *Genome Biol*, PMID 27268795). The types this toolkit touches:

| Type | What it is | Which tools see it |
|---|---|---|
| **missense** | one amino acid swapped for another | AlphaMissense, EVE, ESM1b, REVEL, PrimateAI |
| **synonymous** | DNA changes, protein unchanged (can still break splicing) | splice tools, CADD |
| **inframe indel** | in-frame insertion/deletion of whole codons | (partially) missense tools |
| **splice donor / acceptor** | hits the canonical GT…AG splice signal | SpliceAI, Pangolin, CADD |
| **deep-intronic** | far inside an intron; can create a cryptic splice site | SpliceAI, Pangolin |
| **pLoF** (stop-gain / frameshift / splice) | predicted loss of function — truncates/abolishes the protein | (not scored by missense tools) |

**Missense tools are blind to splice / synonymous / non-coding variants** — which
is exactly why the project has a separate splice analysis (A2, notebooks 09–11).

## To swap in REAL data for the demo tools

Each tool notebook has a *"How to get the REAL data"* section. In short:

- **Pangolin** (10) — run locally (needs GPU)

Already REAL, no swap needed: gnomAD (01), AlphaMissense (02), **EVE** (03,
`data/eve_cftr_2021-08.csv`), ClinVar (07), **CFTR2** (08,
`data/cftr2_2026-01-30.csv`), CADD (11, live API).

Join each on `protein_variant` (missense) or genomic `chr/pos/ref/alt` (splice),
set the loader's `source` to `"REAL"`, and re-run notebook 12.